# RunKeeper activities → `prod.bronze.raw_runkeeper_activities`

Lands the activity log from RunKeeper's export, one row per activity:

```
/Volumes/prod/bronze/stage/runkeeper/activity_logs/cardioActivities.csv   the full history
```

- Every CSV with `cardioActivities` in its name (any case, any subfolder) is read. A newer full export can sit
  beside an older one: each **Activity Id** is kept once, from the CSV with the newest file time.
- **Bronze stays raw:** every value is the text RunKeeper wrote. The units in the headers ("Distance (mi)" or
  "(km)", "Climb (ft)" or "(m)", "Average Speed (mph)" or "(km/h)") become `distance_unit`, `climb_unit` and
  `speed_unit`; a column this notebook doesn't know is kept in `_extra` (JSON).
- **Only new or changed activities are written** (MERGE on `activity_id`, compared by `_row_hash`), each with a
  fresh `_ingested_at`, so the app loads only what changed.
- The table has **Change Data Feed** on and `activity_id` as its **primary key**, as a Lakebase synced table needs.

The tracks are landed by the companion notebook, `runkeeper_gpx_to_bronze`. The app matches them up
(docs/runkeeper-lakebase.md).

**Parameters** (widgets, or the job task's parameters): `activities_dir`, `table`, `dry_run` (`true` counts what
would be written and writes nothing).

In [ ]:
import csv
import hashlib
import io
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

ACTIVITIES_DIR = "/Volumes/prod/bronze/stage/runkeeper/activity_logs"
TABLE = "prod.bronze.raw_runkeeper_activities"
KEY = "activity_id"

# RunKeeper's headers → bronze columns. A header's unit, when it has one, is kept in the *_unit column.
HEADERS = {
    "activity id": "activity_id", "date": "activity_date", "type": "type", "route name": "route_name",
    "distance": "distance", "duration": "duration", "average pace": "average_pace", "average speed": "average_speed",
    "calories burned": "calories_burned", "climb": "climb", "average heart rate": "average_heart_rate_bpm",
    "friend's tagged": "friends_tagged", "friends tagged": "friends_tagged", "notes": "notes", "gpx file": "gpx_file",
}
UNITS = {"distance": "distance_unit", "climb": "climb_unit", "average_speed": "speed_unit"}
CSV_COLUMNS = ["activity_id", "activity_date", "type", "route_name", "distance", "distance_unit", "duration",
               "average_pace", "average_speed", "speed_unit", "calories_burned", "climb", "climb_unit",
               "average_heart_rate_bpm", "friends_tagged", "notes", "gpx_file"]
COLUMNS = CSV_COLUMNS + ["_extra", "_source_file", "_file_modified_at", "_row_hash", "_ingested_at"]
TYPES = {"_file_modified_at": "timestamp", "_ingested_at": "timestamp"}
DDL = f"""CREATE TABLE IF NOT EXISTS {{table}} (
    activity_id STRING NOT NULL COMMENT 'The Activity Id from RunKeeper',
    {", ".join(f"{c} STRING" for c in CSV_COLUMNS[1:])},
    _extra STRING COMMENT 'CSV columns this notebook does not know, as JSON',
    _source_file STRING,
    _file_modified_at TIMESTAMP,
    _row_hash STRING COMMENT 'What the row was written from; an unchanged activity is not written again',
    _ingested_at TIMESTAMP COMMENT 'When this row was last written',
    CONSTRAINT raw_runkeeper_activities_pk PRIMARY KEY (activity_id)
) USING DELTA
COMMENT 'RunKeeper activities (cardioActivities.csv); landed by the runkeeper_activities_to_bronze notebook'
TBLPROPERTIES (delta.enableChangeDataFeed = true)"""

In [ ]:
def _files(folder, suffix):
    """Every file under folder (any depth) whose name ends with suffix, case aside: [(path, size, modified_at)]."""
    out = []
    for root, _dirs, names in os.walk(folder):
        for name in names:
            if name.lower().endswith(suffix):
                path = os.path.join(root, name)
                st = os.stat(path)
                out.append((path, st.st_size, datetime.fromtimestamp(st.st_mtime, tz=timezone.utc)))
    return out


def existing_hashes(spark, table, key, create=True):
    """{key: _row_hash} already landed; the table is made first unless this is a dry run."""
    if create:
        spark.sql(DDL.format(table=table))
    elif not spark.catalog.tableExists(table):
        return {}
    return {r[key]: r["_row_hash"] for r in spark.table(table).select(key, "_row_hash").collect()}


def merge(spark, table, key, rows, prepare=lambda r: r, batch=200):
    """Upsert rows on key, a batch at a time; prepare(row) gives the row as written (e.g. reads its file)."""
    from pyspark.sql import functions as F
    from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType
    types = {"long": LongType(), "timestamp": TimestampType()}
    schema = StructType([StructField(c, types.get(TYPES.get(c), StringType())) for c in COLUMNS if c != "_ingested_at"])
    for i in range(0, len(rows), batch):
        part = [prepare(r) for r in rows[i:i + batch]]
        df = spark.createDataFrame([[r.get(f.name) for f in schema.fields] for r in part], schema)
        df.withColumn("_ingested_at", F.current_timestamp()).createOrReplaceTempView("runkeeper_batch")
        spark.sql(f"""MERGE INTO {table} t USING runkeeper_batch s ON t.{key} = s.{key}
                      WHEN MATCHED AND t._row_hash <> s._row_hash THEN UPDATE SET *
                      WHEN NOT MATCHED THEN INSERT *""")


def setting(name, default, label):
    """A notebook widget (a job task parameter overrides it), or outside Databricks an environment variable."""
    try:
        dbutils.widgets.text(name, default, label)              # noqa: F821  (Databricks provides dbutils)
        return dbutils.widgets.get(name) or default             # noqa: F821
    except NameError:
        return os.environ.get(name.upper(), default)

def column(header):
    """A RunKeeper header as (bronze column, unit or None): 'Distance (mi)' → ('distance', 'mi')."""
    m = re.match(r"\s*(.*?)\s*(?:\(([^)]*)\))?\s*$", header or "")
    name, unit = m.group(1).lower(), m.group(2)
    known = HEADERS.get(name)
    return known, (unit.strip() if unit and known in UNITS else None)


def read_csv(text, source_file, modified_at):
    """One cardioActivities CSV's rows as bronze dicts."""
    reader = csv.reader(io.StringIO(text.lstrip("\ufeff")))
    headers = next(reader, None) or []
    mapped = [column(h) for h in headers]
    rows = []
    for values in reader:
        if not any(v.strip() for v in values):
            continue
        row = {c: None for c in CSV_COLUMNS}
        extra = {}
        for header, (name, unit), value in zip(headers, mapped, values):
            if name:
                row[name] = value.strip() or None
                if unit:
                    row[UNITS[name]] = unit
            elif value.strip():
                extra[header] = value
        if not row["activity_id"]:
            # Older exports may lack ids: the start time and type name the activity.
            if not row["activity_date"]:
                continue
            row["activity_id"] = f"{row['activity_date']}|{row['type'] or ''}"
        row["_extra"] = json.dumps(extra, sort_keys=True) if extra else None
        row["_source_file"] = source_file
        row["_file_modified_at"] = modified_at
        rows.append(row)
    return rows


def read_activities(folder):
    """Every activity in the cardioActivities CSVs under folder: one per activity_id, the newest file winning."""
    found = [f for f in _files(folder, ".csv") if "cardioactivities" in os.path.basename(f[0]).lower()]
    latest = {}
    for path, _size, modified_at in sorted(found, key=lambda f: (f[2], f[0])):
        for row in read_csv(Path(path).read_text(encoding="utf-8-sig", errors="replace"), path, modified_at):
            latest[row["activity_id"]] = row
    return list(latest.values()), [f[0] for f in found]


def row_hash(row):
    basis = [row.get(c) for c in CSV_COLUMNS] + [row.get("_extra")]
    return hashlib.sha256(json.dumps(basis).encode()).hexdigest()


def plan(activities, existing):
    """The activities to write (new or changed) and a summary. existing: {activity_id: _row_hash} already landed."""
    write = [{**a, "_row_hash": row_hash(a)} for a in activities]
    write = [a for a in write if existing.get(a["activity_id"]) != a["_row_hash"]]
    return write, {"activities": len(activities), "with_gpx_file": sum(1 for a in activities if a.get("gpx_file")),
                   "new_or_changed": len(write), "unchanged": len(activities) - len(write)}


def run(spark, activities_dir=ACTIVITIES_DIR, table=TABLE, dry_run=False):
    activities, files = read_activities(activities_dir)
    if not files:
        raise FileNotFoundError(f"No cardioActivities CSV under {activities_dir}")
    rows, summary = plan(activities, existing_hashes(spark, table, KEY, create=not dry_run))
    if rows and not dry_run:
        merge(spark, table, KEY, rows)
    return {"table": table, "csv_files": files, "dry_run": dry_run, **summary}

## Run

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
summary = run(spark, setting("activities_dir", ACTIVITIES_DIR, "Activity log folder"), setting("table", TABLE, "Bronze table"), dry_run=setting("dry_run", "false", "Dry run (true: count only)").lower() == "true")
print(json.dumps(summary, indent=2))
try:
    dbutils.notebook.exit(json.dumps(summary))                   # noqa: F821  (the job run's output)
except NameError:
    pass